In [36]:
import pandas as pd
import numpy as np


In [37]:
path = 'Final_Data/domain.csv'
path_nsw = 'Final_Data/nsw.csv'
pd.set_option('display.width', 0)
# pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 50)

In [38]:
df = pd.read_csv(path)
df_nsw = pd.read_csv(path_nsw)

### 1. Price per square meter


To compare property prices more fairly by using price per area.

In [39]:
df_nsw['price_per_m2'] = df_nsw['Purchase price'] / df_nsw[df_nsw['Area'] > 0]['Area']
pricem2_per_suburb = df_nsw.groupby('Property locality')['price_per_m2'].mean().sort_values(ascending=False).head(10)

### 2. Repeated street transactions

To find streets where properties are sold many times, showing active or popular areas.

In [40]:
df_nsw.groupby('Property street name').size().sort_values(ascending=False).head(10)

Property street name
nancarrow ave    15638
pacific hwy       7854
george st         7842
pitt st           6744
gladstone st      6075
princes hwy       4371
church st         4091
king st           3452
victoria st       3396
william st        3240
dtype: int64

### 3. Average Property Price by Postcode

To see which postal areas have higher or lower property prices.


In [41]:
df_nsw.groupby('Property post code')['Purchase price'].mean().sort_values(ascending=False).head(10)

Property post code
2555.0    7.042619e+06
2556.0    4.063826e+06
2748.0    3.548478e+06
2108.0    3.452318e+06
2063.0    3.434728e+06
2030.0    3.413127e+06
2092.0    3.260379e+06
2104.0    3.245601e+06
2110.0    3.112238e+06
2175.0    3.110158e+06
Name: Purchase price, dtype: float64

### 4. Repeat-Sales Price Analysis

To check how prices change when the same property is sold again later.

In [42]:
df_nsw.groupby('Sale counter')['Purchase price'].agg(['mean', 'median', 'count']).head(10).reset_index()

,Sale counter,mean,median,count
0,1,780606.646722,550000.0,46479
1,2,779720.788998,550000.0,44265
2,3,809471.651105,565000.0,41984
3,4,827236.557830,585000.0,39564
4,5,860863.560213,607500.0,37268
5,6,880415.856326,626000.0,35156
6,7,905025.113546,649000.0,33308
7,8,921829.511574,658500.0,31708
8,9,950203.909728,676000.0,30253
9,10,968498.439913,685000.0,29008


### 5. Monthly Median Price Trend

To see how much price changes per month


In [52]:
dfa = df_nsw.copy()
dfa['Contract date'] = pd.to_datetime(df_nsw['Contract date'], errors='coerce')
dfa = dfa.dropna(subset=['Contract date', 'price_per_m2'])
dfa = dfa.sort_values('Contract date').set_index('Contract date')
mean_change = dfa['price_per_m2'].resample('M').median().pct_change() * 100
mean_change.sort_index(ascending=False).head(10)

C:\Users\DucVu\AppData\Local\Temp\ipykernel_20416\1976304432.py:5: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  mean_change = dfa['price_per_m2'].resample('M').median().pct_change() * 100
C:\Users\DucVu\AppData\Local\Temp\ipykernel_20416\1976304432.py:5: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  mean_change = dfa['price_per_m2'].resample('M').median().pct_change() * 100


Contract date
2025-08-31   -27.254532
2025-07-31   -12.115901
2025-06-30    -9.731330
2025-05-31    -3.305030
2025-04-30    -2.021725
2025-03-31     8.203814
2025-02-28    18.196545
2025-01-31    -2.898535
2024-12-31   -15.779242
2024-11-30     2.204580
Freq: -1ME, Name: price_per_m2, dtype: float64

### 6. Average property price by type and state

To compare property prices between states and property types.

In [ ]:
price_of_type_in_state = df.pivot_table(
    values='Price',
    index='Type',
    columns='state',
    aggfunc='mean',
    margins=True,
    margins_name='Total'
)
price_of_type_in_state

state,act,nsw,nt,qld,sa,tas,vic,wa,Total
Type,,,,,,,,,
Apartment / Unit / Flat,5.730860e+05,9.051775e+05,342251.000000,1.029890e+06,6.387375e+05,5.587778e+05,6.812062e+05,7.075882e+05,8.168276e+05
Block of units,NaN,1.533033e+06,341875.000000,1.032023e+06,9.710227e+05,8.810000e+05,9.338261e+05,1.241152e+06,1.192111e+06
Development site,8.200000e+05,1.462226e+06,455000.000000,1.074584e+06,4.747432e+05,4.302857e+05,1.209996e+06,6.828269e+05,1.180771e+06
Duplex,1.205333e+06,1.388329e+06,509000.000000,9.584072e+05,3.240053e+05,5.765667e+05,1.281619e+06,6.718273e+05,1.142365e+06
New apartments / off the plan,6.624195e+05,1.169479e+06,477500.000000,1.368529e+06,7.850625e+05,3.650000e+06,1.392051e+06,9.322760e+05,1.150536e+06
New home designs,NaN,9.028899e+05,NaN,9.293557e+05,1.096441e+06,8.345252e+05,7.441355e+05,9.853000e+05,8.017531e+05
New house and land,1.236143e+06,1.219694e+06,742500.000000,9.432492e+05,1.085194e+06,6.502153e+05,8.723078e+05,8.796776e+05,1.035194e+06
New land,6.798333e+05,7.375489e+05,NaN,4.392373e+05,2.807333e+05,2.731250e+05,4.731985e+05,4.335479e+05,5.444850e+05
Penthouse,1.172625e+06,1.762315e+06,NaN,1.665091e+06,9.173214e+05,1.152500e+06,1.083450e+06,1.476500e+06,1.596467e+06


### 7. Percentage of properties with parking by state


To check how common parking spaces are in each state.

In [ ]:
df.assign(has_parking = df['Parking'] > 0).groupby('state')['has_parking'].mean().sort_values(ascending=False)

state
act    0.901786
nt     0.854545
qld    0.773810
nsw    0.752306
vic    0.746605
wa     0.678654
sa     0.600998
tas    0.554545
Name: has_parking, dtype: float64

### 8. Comparison of average property prices between those with and without parking

To see how parking affects property price.

In [57]:
df['has_parking'] = df['Parking'] > 0
df.pivot_table(
    index='state',
    columns='has_parking',
    values='Price',
    aggfunc='mean'
)

has_parking,False,True
state,,
act,5.274826e+05,7.480610e+05
nsw,1.047989e+06,1.234091e+06
nt,3.822500e+05,4.434948e+05
qld,5.350867e+05,9.762119e+05
sa,3.435814e+05,7.649673e+05
tas,3.284186e+05,7.060714e+05
vic,5.704986e+05,8.675321e+05
wa,5.289143e+05,7.436267e+05


### 9. Coefficient of variation of each property type by state

To measure how much prices change inside each property type.

In [ ]:
df.pivot_table(
    values='Price',
    index='state',
    columns='Type',
    aggfunc=lambda x: x.std() / x.mean()
)

Type,Apartment / Unit / Flat,Block of units,Development site,Duplex,New apartments / off the plan,New home designs,New house and land,New land,Penthouse,Retirement Living,Semi-detached,Studio,Terrace,Townhouse,Vacant land,Villa
state,,,,,,,,,,,,,,,,
act,0.341363,NaN,0.370298,0.351506,0.502976,NaN,0.160247,0.220691,0.479676,NaN,0.316079,0.261809,0.303950,0.381210,0.130241,NaN
nsw,0.553458,0.761223,0.924993,0.513167,0.826562,0.520704,0.587435,0.788024,0.689353,0.663421,0.460129,0.397139,0.513057,0.512890,0.875260,0.411498
nt,0.352388,0.212794,0.145371,0.299917,0.170298,NaN,0.042855,NaN,NaN,NaN,0.379897,0.090516,NaN,0.237249,NaN,NaN
qld,0.638753,0.702963,0.791681,0.492598,0.832323,0.397352,0.602128,0.842148,0.722352,0.513216,0.596762,0.433043,0.593919,0.354537,1.082714,0.783778
sa,0.430955,0.559686,0.832760,0.284752,0.492652,1.479334,0.388841,0.550609,0.610292,0.160958,1.252696,1.021729,0.452258,0.247730,0.999262,0.614153
tas,0.402390,0.558932,0.783630,0.311445,NaN,0.201287,0.304089,0.557348,0.629695,0.267000,NaN,0.366063,0.687628,0.183008,0.471441,0.332529
vic,0.632496,0.536676,0.851984,0.982868,0.792113,0.392878,0.682034,1.263119,0.802466,0.418486,0.588090,0.694983,0.731140,0.467237,0.967858,0.444495
wa,0.509676,0.848772,0.680411,0.254190,0.766546,0.877722,0.669486,0.536848,0.635986,0.233673,0.306616,0.317019,0.733545,0.390782,0.734989,0.237721


### 10. Dominant property type by suburb

To find which property type is most common in each suburb.

In [ ]:
counts = df.groupby(['suburb', 'Type']).size().unstack(fill_value=0)
dominant_type = counts.idxmax(axis=1)
dominant_count = counts.max(axis=1)
top10 = dominant_count.sort_values(ascending=False).head(10)
top10_df = pd.DataFrame({
    'Dominant_Type': dominant_type[top10.index],
    'Count': top10
})
top10_df

,Dominant_Type,Count
suburb,,
paddington,Terrace,154
surry hills,Terrace,83
darlinghurst,Terrace,61
potts point,Studio,59
elizabeth bay,Studio,57
melbourne,Studio,54
schofields,New apartments / off the plan,48
shell cove,New land,47
newcastle,New apartments / off the plan,41
